<a href="https://colab.research.google.com/github/Mrinal0044/Machine-Learning/blob/main/column_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ordinal categorical data -> OE

nominal categorical data -> OHE

there will be different types of data columns and we will have to apply different encoding for each resulting in many numpy arrays.

to avoid this we make COLUMN TRANSFORMER.


In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_csv("/content/covid_toy.csv")

In [4]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [6]:
X_train

,age,gender,fever,cough,city
47,18,Female,104.0,Mild,Bangalore
24,13,Female,100.0,Strong,Kolkata
27,33,Female,102.0,Strong,Delhi
73,34,Male,98.0,Strong,Kolkata
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
8,19,Female,100.0,Strong,Bangalore
38,49,Female,101.0,Mild,Delhi
89,46,Male,103.0,Strong,Bangalore
78,11,Male,100.0,Mild,Bangalore


1. NOT USING COLUMN TRANSFORMER

-> adding simple imputer to fever to fill all missing values

-> ordinal encoding in cough

-> one hot encoding in gender and city

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [11]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [12]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [15]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [16]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [17]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

2. USING COLUMN TRANSFORMER

In [18]:
from sklearn.compose import ColumnTransformer

In [20]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

remainder = passthrough means it will not affect the columns which is not getting transformed

drop first is done to remove collinearity

In [21]:
transformer.fit_transform(X_train).shape

(80, 7)

In [22]:
transformer.transform(X_test).shape

(20, 7)